# 00c — Prepare modern large-schema benchmarks (R1.1 / R3)

Converts Spider 2.0-lite, BEAVER, and Spider 2.0 into the pipeline's `data/<Dkey>/` layout so notebooks 01–09 run unchanged. **The Spider 2.0-lite loader below was validated on the real `xlang-ai/Spider2` repo** (135 local SQLite instances, 30 DBs, 35–352 columns/db). These benchmarks ship no curated glossary, so they also test generalization to new databases (R3).

## Spider 2.0-lite (D4) — validated loader
Clone `xlang-ai/Spider2`, set `SPIDER2_LITE`, and (for execution) download the local `.sqlite` files per its README into `resource/databases/spider2-localdb/`.

In [1]:
# --- Spider 2.0-lite (SQLite subset) preparation --------------------------------
# VALIDATED on the real repo (xlang-ai/Spider2): 135 `local*` SQLite instances
# across 30 databases, schema sizes 35-352 columns (median ~106) -- an
# enterprise-scale stress test. 24 instances ship a gold SQL (recall-scoreable);
# the other 111 are evaluated by Spider 2.0's result-based evaluator (gold result
# CSVs + the .sqlite files, which you download per the spider2-lite README).
import json, re, csv
from pathlib import Path
import sqlglot
from sqlglot import exp

SPIDER2_LITE = Path("path/to/Spider2/spider2-lite")   # EDIT: cloned xlang-ai/Spider2
SCHEMA_DIR = SPIDER2_LITE/"resource/databases/sqlite"          # per-table JSON + DDL.csv
GOLD_DIR   = SPIDER2_LITE/"evaluation_suite/gold/sql"          # <instance_id>.sql (24 local)
# .sqlite binaries go in resource/databases/spider2-localdb/<db>.sqlite (README download)

def _norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())

def resolve_db_dir(db):
    if (SCHEMA_DIR/db).exists(): return db
    want = _norm(db)
    for d in SCHEMA_DIR.iterdir():
        if d.is_dir() and _norm(d.name) == want: return d.name
    return None

def build_schema_entry(dbdir):
    """Schema entry from per-table JSON (column_names/types/sample_rows) + DDL FKs.
    Works WITHOUT the .sqlite file; sample_rows supply representative values."""
    ddir = SCHEMA_DIR/dbdir; tables, fks = {}, []
    for jf in sorted(ddir.glob("*.json")):
        d = json.loads(jf.read_text()); tname = d.get("table_name") or jf.stem
        sbc = {}
        for row in d.get("sample_rows", [])[:5]:
            for k, v in row.items():
                sbc.setdefault(k, [])
                if v is not None and len(sbc[k]) < 5: sbc[k].append(v)
        cols = {name: {"type": typ or "TEXT", "pk": False, "desc": "",
                       "sample_values": sbc.get(name, [])}
                for name, typ in zip(d.get("column_names", []), d.get("column_types", []))}
        tables[tname] = {"columns": cols}
    ddl = ddir/"DDL.csv"
    if ddl.exists():
        for r in csv.DictReader(ddl.open()):
            t, sql = r["table_name"], r["DDL"]
            for m in re.finditer(r'FOREIGN\s+KEY\s*\(([^)]+)\)\s*REFERENCES\s+"?(\w+)"?\s*\(([^)]+)\)', sql, re.I):
                fks.append({"from": f"{t}.{m.group(1).split(',')[0].strip()}",
                            "to": f"{m.group(2)}.{m.group(3).split(',')[0].strip()}"})
    # NOTE: most Spider 2.0-lite analytics DBs declare NO foreign keys (only 1/30 here).
    # This is the "new database without curated metadata" regime (Reviewer 3): graph
    # traversal must fall back on mined/name-based joins. mine_join_patterns() on the
    # gold/query log fills `join_patterns`; otherwise leave it empty and report the
    # cold-start behaviour.
    return {"db_id": dbdir, "tables": tables, "foreign_keys": fks,
            "glossary": [], "join_patterns": []}

def sqlglot_schema(entry):
    return {t: {c: ci["type"] for c, ci in ti["columns"].items()}
            for t, ti in entry["tables"].items()}

def resolve_gold_columns(cset, sch):     # same resolver as the 00b patch (R2.3-i)
    universe, by_name = set(), {}
    for t, cols in sch.items():
        for c in cols:
            universe.add(f"{t.lower()}.{c.lower()}"); by_name.setdefault(c.lower(), []).append(t.lower())
    resolved, dropped = set(), 0
    for c in cset:
        c = str(c).strip().lower()
        if not c or c == "*" or c.endswith(".*"): dropped += 1; continue
        if "." in c:
            if c in universe: resolved.add(c)
            else:
                o = by_name.get(c.split(".")[-1], [])
                resolved.add(f"{o[0]}.{c.split('.')[-1]}") if len(o)==1 else (dropped:=dropped+1)
        else:
            o = by_name.get(c, [])
            resolved.add(f"{o[0]}.{c}") if len(o)==1 else (dropped:=dropped+1)
    return sorted(resolved), dropped

def extract_gold(gold_sql, real_tables):
    try: tree = sqlglot.parse_one(gold_sql, read="sqlite")
    except Exception: return None, None, "parse_fail"
    cte = {c.alias_or_name.lower() for c in tree.find_all(exp.CTE)}
    tabs = {t.name for t in tree.find_all(exp.Table)
            if t.name and t.name.lower() not in cte and t.name.lower() in real_tables}
    cols = set()
    for col in tree.find_all(exp.Column):
        if col.table and col.table.lower() not in cte: cols.add(f"{col.table}.{col.name}")
        elif not col.table and col.name: cols.add(col.name)
    return tabs, cols, None

def prepare_spider2_lite(out_dir="data/D4"):
    rows = [json.loads(l) for l in (SPIDER2_LITE/"spider2-lite.jsonl").open()]
    loc = [r for r in rows if str(r["instance_id"]).startswith("local")]
    cache, out = {}, []
    for r in loc:
        dbdir = resolve_db_dir(r["db"])
        if dbdir is None: continue
        cache.setdefault(dbdir, build_schema_entry(dbdir))
        entry = cache[dbdir]; real = {t.lower() for t in entry["tables"]}
        rec = {"qid": r["instance_id"], "db_id": dbdir, "question": r["question"],
               "gold_sql": "", "gold_tables": [], "gold_columns": [],
               "recall_scoreable": False, "evidence": ""}
        gf = GOLD_DIR/f"{r['instance_id']}.sql"
        if gf.exists():
            tabs, cols, err = extract_gold(gf.read_text(), real)
            if not err:
                gc, dropped = resolve_gold_columns(cols, sqlglot_schema(entry))
                rec.update(gold_sql=gf.read_text(), gold_tables=sorted(tabs),
                           gold_columns=gc, n_gold_cols_dropped=dropped, recall_scoreable=True)
        out.append(rec)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    Path(out_dir, "questions.json").write_text(json.dumps(out, indent=1))
    Path(out_dir, "schema.json").write_text(json.dumps(list(cache.values()), indent=1))
    ncols = [sum(len(t["columns"]) for t in e["tables"].values()) for e in cache.values()]
    print(f"Spider2-lite: {len(out)} questions, {len(cache)} DBs, "
          f"columns/db min={min(ncols)} median={sorted(ncols)[len(ncols)//2]} max={max(ncols)}; "
          f"recall-scoreable={sum(r['recall_scoreable'] for r in out)}")
    return out

# prepare_spider2_lite("data/D4")   # <- run after setting SPIDER2_LITE and downloading .sqlite

## Optional — build schema directly from a `.sqlite` file

In [2]:
# --- Optional: build the schema straight from a .sqlite file (exact types/FKs/values)
# Use this instead of the JSON path once you have downloaded the local DBs, or for
# any SQLite benchmark (e.g. a SQLite export of BEAVER).
import sqlite3
def sqlite_schema_entry(db_id, sqlite_path):
    con = sqlite3.connect(str(sqlite_path)); cur = con.cursor()
    tables, fks = {}, []
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")
    for (t,) in cur.fetchall():
        cols = {}
        for _, name, ctype, _, _, pk in cur.execute(f'PRAGMA table_info("{t}")'):
            samples = [v for (v,) in cur.execute(
                f'SELECT DISTINCT "{name}" FROM "{t}" WHERE "{name}" IS NOT NULL LIMIT 5')]
            cols[name] = {"type": ctype or "TEXT", "pk": bool(pk), "desc": "", "sample_values": samples}
        tables[t] = {"columns": cols}
        for _, _, rt, frm, to, *_ in cur.execute(f'PRAGMA foreign_key_list("{t}")'):
            fks.append({"from": f"{t}.{frm}", "to": f"{rt}.{to}"})
    con.close()
    return {"db_id": db_id, "tables": tables, "foreign_keys": fks, "glossary": [], "join_patterns": []}

## BEAVER (D5) and Spider 2.0 full (D6)

In [3]:
# --- BEAVER (D5) and Spider 2.0 full (D6) ---------------------------------------
# BEAVER (arXiv:2409.02038): enterprise warehouse, ~812 tables, access-gated. Once
# obtained, build each schema with sqlite_schema_entry() (export the referenced
# subset to SQLite) or add a warehouse execution backend as for D6. glossary stays
# [] unless BEAVER ships business descriptions. This is the strongest large-schema
# test of the savings-vs-size claim.
#
# Spider 2.0 full (D6, Snowflake/BigQuery): parse schema as above, and add an
# execution backend run_sql_warehouse(db_id, sql) using the Snowflake/BigQuery
# client; branch notebook 02's execution on dataset=="D6". Report D6 as a
# schema-selection stress test (recall/token scaling at 1000+ columns), NOT as an
# agentic-SOTA EX claim, since the pipeline is single-call. Credentials go in
# evaluation_suite/{snowflake,bigquery}_credential.json per the Spider2 repo.

After preparing a benchmark, add it to notebook 02's `FULL_PLAN`, e.g. `("D4", ["C1","C4","C5_L1","C5_L2"], ["STRONG"])`, and re-run 01→02→03..09. Report token savings and recall vs schema size across D1<D2<D4<D5<D6 (the savings-grow-with-scale claim gets its strongest test here).